# LLM/RAG Initialization

In [1]:
import pickle

# Load data for the RAG (processed_data.pkl)
with open('processed_data.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

In [2]:
from langchain import PromptTemplate, LLMChain, HuggingFacePipeline
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re
from rag_source import *
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  #This is directory

# Load the LLaMA model for text generation
model_id = "meta-llama/Meta-Llama-3-8B"

llama_pipeline = pipeline(
   "text-generation", 
   model=model_id, 
   model_kwargs={"torch_dtype": torch.bfloat16}, 
   device_map="auto",
   temperature=0.3,
   top_p=0.9,
   max_length=1000
)

llm = HuggingFacePipeline(pipeline=llama_pipeline)

# local_model = "llama_8B"

# pipeline = pipeline(
#     "text-generation", 
#     model=local_model, #model_id, 
#     model_kwargs={"torch_dtype": torch.bfloat16}, 
#     device_map="auto",
#     temperature=0.2,  # Control randomness (lower values = more focused responses)
#     top_p=0.9,  # Filter unlikely words
#     truncation=True,
#     max_length=512,
#     max_new_tokens=1024
# )

# llm = HuggingFacePipeline(pipeline=pipeline)

# Define a PromptTemplate that accepts prompt_text as an input
prompt_template = PromptTemplate(
    input_variables=["prompt_text"],
    template="{prompt_text}"
)

# Initialize LLMChain with the LLM and prompt template
llm_chain = LLMChain(
    prompt=prompt_template,
    llm=llm
)

# Function to directly analyze the provided text prompt
def analyze_text_prompt(query, processed_data):
    # Pass the prompt text to LLMChain as a dictionary

    prompt_text = process_query(query, processed_data, embedding_model) + " The correct answer is, "

    inputs = {
        "prompt_text": prompt_text
    }
    
    # Run the LLM chain and get the result
    result = llm_chain.run(inputs)

    return prompt_text, result


ModuleNotFoundError: No module named 'langchain'

In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Pass the query

In [4]:
import torch
import gc

gc.collect()

# Clear the PyTorch CUDA memory cache
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# Optionally, reset PyTorch memory stats (useful for debugging)
torch.cuda.reset_peak_memory_stats()

In [5]:
import torch
print(f"Allocated memory: {torch.cuda.memory_allocated() / 1024 ** 3:.2f} GB")
print(f"Reserved memory: {torch.cuda.memory_reserved() / 1024 ** 3:.2f} GB")
print(f"Max memory allocated: {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GB")

Allocated memory: 12.84 GB
Reserved memory: 12.97 GB
Max memory allocated: 12.84 GB


In [6]:
query = "Which replacement policy yields the lowest overall miss rate on the lbm workload?"
#For the cache access with PC 0x403a85 and address 0x35e798a637f on the bzip workload with PARROT replacement policy, does the cache hit or miss?
prompt, response = analyze_text_prompt(query, loaded_data)

# Remove the prompt_text from the beginning of result
if response.startswith(prompt):
    response = response[len(prompt):]

print("\nPrompt: ", prompt)
print("\nResponse: ", response)

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Finetuned model

In [14]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  #This is directory

In [18]:
import pickle
from langchain import PromptTemplate, LLMChain, HuggingFacePipeline
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re
from rag_source import *
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain.llms import HuggingFacePipeline

model_id = "llama"

config = PeftConfig.from_pretrained("outputs/final_checkpoint/")
model = AutoModelForCausalLM.from_pretrained(model_id)
model = PeftModel.from_pretrained(model, "outputs/final_checkpoint/")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
pipeline = pipeline(
    "text-generation",
    model=model, #model_id, 
    model_kwargs={"torch_dtype": torch.bfloat16},
    tokenizer=tokenizer,
    device_map="auto",
    temperature=0.2,  # Control randomness (lower values = more focused responses)
    top_p=0.9,  # Filter unlikely words
    truncation=True,
    max_length=500,
    max_new_tokens=150
)

llm = HuggingFacePipeline(pipeline=pipeline)

# Define a PromptTemplate that accepts prompt_text as an input
prompt_template = PromptTemplate(
    input_variables=["prompt_text"],
    template="{prompt_text}"
)

# Initialize LLMChain with the LLM and prompt template
llm_chain = LLMChain(
    prompt=prompt_template,
    llm=llm
)

# Function to directly analyze the provided text prompt
def analyze_text_prompt(query, processed_data):
    # Pass the prompt text to LLMChain as a dictionary

    prompt_text = process_query(query, processed_data, embedding_model) + " The correct answer is, "

    inputs = {
        "prompt_text": prompt_text
    }
    
    # Run the LLM chain and get the result
    result = llm_chain.run(inputs)

    return prompt_text, result

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', '

## Pass The Query

In [19]:
query = "What is the miss rate for PC 0x4037ba on the mcf workload with PARROT replacement policy?"
#For the cache access with PC 0x403a85 and address 0x35e798a637f on the bzip workload with PARROT replacement policy, does the cache hit or miss?
prompt, response = analyze_text_prompt(query, loaded_data)

# Remove the prompt_text from the beginning of result
if response.startswith(prompt):
    response = response[len(prompt):]

print("\nPrompt: ", prompt)
print("\nResponse: ", response)

Both `max_new_tokens` (=150) and `max_length`(=500) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt:  You are an expert in computer architecture and your job is to answer questions given data from cache traces. Base your response on the following data and your knowledge of computer architecture.

Workloads involved:
- mcf is a C-based benchmark derived from MCF, a program for single-depot vehicle scheduling in public transportation. It uses integer arithmetic to solve large-scale minimum-cost flow problems with a network simplex algorithm. The goal is to minimize fleet size and operational costs by scheduling timetabled trips efficiently. The benchmark employs MCF Version 1.2, integrating it with column generation to accelerate the solution process, relying heavily on pointer and integer arithmetic.

Policies involved:
- MLP: learned policy using a simple multi-layer perceptron to approximate belady's optimal policy
- PARROT: learned policy using attention to approximate belady's optimal policy
- Belady: optimal replacement policy. Evict the cache line with highest reuse dist